In [1]:
#load libraries
import os
import io
import pandas as pd
import datetime as dt
import msoffcrypto
import openpyxl
import tkinter as tk
from tkinter import filedialog

In [ ]:
#set up function to select directory and list files in it

def select_directory():
    root = tk.Tk()
    root.withdraw() #hide the root window
    root.attributes('-topmost', True)

    try:
        cwd = filedialog.askdirectory(parent=root, title="Select a Folder")

        if cwd:
            cwd = cwd + "/"
            files = os.listdir(cwd)

            print(f"Selected folder: {cwd}")
            print("Files:", files)

            return cwd, files
        else:
            print("No folder selected.")
            return None, None

    finally:
        root.destroy() #always destroy the hidden root window


if __name__ == "__main__":
    cwd, files = select_directory()


In [3]:
#display files in current working directory, as a quick visual check
files

['.git',
 'File1.xlsx',
 'File2.xlsx',
 'File3.xlsx',
 'File_Aggregator.ipynb',
 'LICENSE',
 'README.md']

In [4]:
#count of Excel files
count = 0

for file in files:
    if file.endswith('.xlsx'):
        count = count + 1
print(count)

3


In [5]:
###**********THIS IS FOR NON-PASSWORD PROTECTED EXCEL FILES**********###

#combines all non-protected files, selecting only the 'Input Information' tab(s)
frames = []
for file in files:
    if file.endswith('.xlsx'):
        path = os.path.join(cwd, file)
        frames.append(pd.read_excel(path, sheet_name='Input Information'))
combined_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
combined_df.head()

,Project,Date,Status
0,Project 1,2020-04-05,Approved
1,Project 2,2020-04-06,Approved
2,Project 3,2020-04-07,Approved
3,Project 4,2020-04-08,Approved
4,Project 5,2020-04-09,Approved


In [13]:
###**********THIS IS FOR PASSWORD PROTECTED EXCEL FILES**********###

#combine all protected files, selecting only the 'Input Information' tab(s)
workbook = pd.DataFrame()
combined_df = pd.DataFrame()

start = dt.datetime.now()

for file in files:
    decrypted_workbook = io.BytesIO()
    workbook = pd.DataFrame()
    if file.endswith('.xlsx'):
        with open(cwd+file, 'rb') as data:
            office_file = msoffcrypto.OfficeFile(data)
            office_file.load_key(password='thisisthepassword') #PASSWORD goes here
            office_file.decrypt(decrypted_workbook)
            workbook = pd.read_excel(decrypted_workbook, sheet_name=2) #reference sheet 2 in workbook, not by sheet name. original value was "Input Information"
            combined_df = pd.concat([combined_df, workbook], ignore_index=True)

end = dt.datetime.now()

print(f'Elapsed Time: {end-start}')

DecryptionError: Document is not encrypted

In [6]:
#display combined data
combined_df

,Project,Date,Status
0,Project 1,2020-04-05,Approved
1,Project 2,2020-04-06,Approved
2,Project 3,2020-04-07,Approved
3,Project 4,2020-04-08,Approved
4,Project 5,2020-04-09,Approved
5,Project 6,2020-04-10,Approved
6,Project 7,2020-04-11,Approved
7,Project 8,2020-04-12,Approved
8,Project 9,2020-04-13,Approved
9,Project 10,2020-04-14,Approved


In [7]:
#clean up combined_df, null values
combined_df.dropna(thresh=2, inplace=True) #require 2 data points in the row to be kept, any less than 2 means the row is dropped

#any true NaNs filled as [blank]
combined_df.fillna("", inplace=True)

In [8]:
#display clean combined data for all files
combined_df

,Project,Date,Status
0,Project 1,2020-04-05,Approved
1,Project 2,2020-04-06,Approved
2,Project 3,2020-04-07,Approved
3,Project 4,2020-04-08,Approved
4,Project 5,2020-04-09,Approved
5,Project 6,2020-04-10,Approved
6,Project 7,2020-04-11,Approved
7,Project 8,2020-04-12,Approved
8,Project 9,2020-04-13,Approved
9,Project 10,2020-04-14,Approved


In [9]:
#create new output file with combined data and save to current working directory
combined_df.to_excel(cwd+'Aggregation_Output.xlsx')